## الكشف عن العمر والوجوه باستخدام التعلم العميق في مكتبة OpenCV

سنقوم هنا بدمج وظائف اكتشاف الوجه والتنبؤ بالعمر لتقديم حل متكامل.

### 1. استيراد المكتبات الضرورية

In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
import os

print("تم استيراد المكتبات بنجاح.")

### 2. تحميل النماذج المدربة مسبقًا

نحتاج إلى نماذج لاكتشاف الوجه والتنبؤ بالعمر. تأكد من توفر هذه الملفات في بيئة العمل الخاصة بك (على سبيل المثال، قم بتحميلها إلى مجلد `/content/` في Colab).

In [ ]:
# لتنزيل الملفات (إذا لم تكن متوفرة بالفعل)
# !wget -N https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt -O opencv_face_detector.pbtxt
# !wget -N https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/opencv_face_detector_uint8.pb -O opencv_face_detector_uint8.pb
# !wget -N https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_20170830/age_deploy.prototxt -O age_deploy.prototxt
# !wget -N https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_20170830/age_net.caffemodel -O age_net.caffemodel

face_proto  =  "opencv_face_detector.pbtxt"
face_model  =  "opencv_face_detector_uint8.pb"
age_proto  =  "age_deploy.prototxt"
age_model  =  "age_net.caffemodel"

# تحميل النماذج
face_net  =  cv2.dnn.readNetFromTensorflow ( face_model , face_proto )
age_net  =  cv2.dnn.readNetFromCaffe ( age_proto , age_model )

print("تم تحميل نماذج اكتشاف الوجه والعمر بنجاح.")

### 3. تعريف الثوابت وقائمة الأعمار

هذه القيم ضرورية لنموذج التنبؤ بالعمر.

In [ ]:
MODEL_MEAN_VALUES = (78.4263377603, 87.7689143744, 114.895847746)
age_list = ['(0-2)', '(4-6)', '(8-12)', '(15-20)', '(25-32)', '(38-43)', '(48-53)', '(60-100)']

print("تم تعريف قيم Mean وقائمة الأعمار.")

### 4. دالة اكتشاف الوجوه (`detect_faces`)

تستخدم هذه الدالة نموذج اكتشاف الوجه لتحديد مواقع الوجوه في الصورة.

In [ ]:
def  detect_faces ( net , frame , confidence_threshold = 0.7 ):
    frame_height  =  frame.shape [ 0 ]
    frame_width  =  frame.shape [ 1 ]
    blob  =  cv2.dnn.blobFromImage ( frame , 1.0 , ( 300 , 300 ) , [ 104 , 117 , 123 ] , False , False )
    net.setInput ( blob )
    detections  =  net.forward ( )
    face_boxes  = []
    for  i  in  range ( detections . shape [ 2 ]):
        confidence  =  detections [ 0 , 0 , i , 2 ]
        if  confidence  >  confidence_threshold :
            x1  =  int ( detections [ 0 , 0 , i , 3 ] *  frame_width )
            y1  =  int ( detections [ 0 , 0 , i , 4 ] *  frame_height )
            x2  =  int ( detections [ 0 , 0 , i , 5 ] *  frame_width )
            y2  =  int ( detections [ 0 , 0 , i , 6 ] *  frame_height )
            face_boxes . append ([ x1 , y1 , x2 , y2 ])
            cv2.rectangle ( frame , ( x1 , y1 ), ( x2 , y2 ), ( 0 , 255 , 0 ) , int ( round ( frame_height / 150 ) ), 8 )
    return  frame , face_boxes

### 5. دالة التنبؤ بالعمر (`predict_age`)

تستخدم هذه الدالة نموذج التنبؤ بالعمر لتصنيف عمر الوجه المقطوع ضمن نطاقات محددة.

In [ ]:
def  predict_age ( face , net ):
    blob  =  cv2.dnn.blobFromImage ( face , 1.0 , ( 227 , 227 ) , MODEL_MEAN_VALUES , swapRB = False )
    net.setInput ( blob )
    age_preds  =  net.forward ( )
    age  =  age_list [ age_preds [ 0 ]. argmax ()]
    return  age

### 6. دالة معالجة الصورة الكاملة (`process_image`)

تجمع هذه الدالة بين اكتشاف الوجه والتنبؤ بالعمر، وتعرض النتائج على الصورة.

In [ ]:
def  process_image ( image_path ):
    frame  =  cv2.imread ( image_path )

    if  frame  is  None :
        print ( f"خطأ: لم يتم العثور على الصورة في { image_path } " )
        return

    frame , face_boxes  =  detect_faces ( face_net , frame )

    for ( x1 , y1 , x2 , y2 ) in  face_boxes :
        # قص منطقة الوجه مع بعض الهامش
        face  =  frame [ max ( 0 , y1 - 20 ) : min ( y2 + 20 , frame.shape [ 0 ] - 1 ) ,
                     max ( 0 , x1 - 20 ) : min ( x2 + 20 , frame.shape [ 1 ] - 1 ) ]

        # التأكد من أن منطقة الوجه ليست فارغة قبل التنبؤ بالعمر
        if face.shape[0] > 0 and face.shape[1] > 0:
            age  =  predict_age ( face , age_net )
            cv2.putText ( frame , f "العمر: { age } " , ( x1 , y1 - 10 ) ,
                       cv2.FONT_HERSHEY_SIMPLEX , 0.8 , ( 0 , 255 , 255 ) , 2 , cv2.LINE_AA )
        else:
            cv2.putText ( frame , "وجه غير صالح" , ( x1 , y1 - 10 ) ,
                       cv2.FONT_HERSHEY_SIMPLEX , 0.8 , ( 0 , 0 , 255 ) , 2 , cv2.LINE_AA )

    cv2_imshow ( frame )
    # cv2.waitKey ( 0 ) # لا نحتاج لهذه في بيئة Colab مع cv2_imshow
    # cv2.destroyAllWindows ( ) # لا نحتاج لهذه في بيئة Colab مع cv2_imshow

### 7. تطبيق عملي

لنجرب الدالة على صورة. يمكنك تحميل صورتك الخاصة أو استخدام صورة تجريبية (مثل `kid1.jpg` المشار إليها في النص الأصلي).

In [ ]:
# لتنزيل صورة تجريبية (إذا لم تكن متوفرة بالفعل)
# !wget -N https://www.thesun.co.uk/wp-content/uploads/2019/02/NINTCHDBPICT000469317208-1.jpg?strip=all&w=1200 -O kid1.jpg

image_path  =  "kid1.jpg"

# التأكد من وجود الملف
if not os.path.exists(image_path):
    print(f"الملف {image_path} غير موجود. يرجى التأكد من تحميله.")
else:
    process_image ( image_path )